# Detection ploidie avec nQuire

#### Snakefile_nQuire

In [ ]:
############################################
# nQuire Snakemake pipeline
############################################

configfile: "config.yaml"

############################################
# Lire table BAM
############################################

samples = {}

with open(config["bam_table"]) as f:
    for line in f:
        bam = line.strip()
        if not bam:
            continue

        ind = bam.split("/")[-1].replace(".sorted.rg.bam", "")
        samples[ind] = {"bam": bam}

INDIVIDUALS = list(samples.keys())

############################################
# Rule all
############################################

rule all:
    input:
        "results/all_lrd_not_denoised.txt",
        "results/all_lrd_denoised.txt",
        expand("results/histo/{ind}.histo.txt", ind=INDIVIDUALS),
        expand("results/histotest/{ind}.histotest.txt", ind=INDIVIDUALS)


############################################
# nQuire create
############################################

rule nquire_create:
    input:
        lambda wc: samples[wc.ind]["bam"]
    output:
        "results/{ind}.base.bin"
    log:
        "results/logs/{ind}_create.log"
    resources:
        mem_mb=32000,
        slurm_partition="workq",
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/nQuire/a990a88

        nQuire create \
            -b {input} \
            -o results/{wildcards.ind}.base 
        """

############################################
# nQuire denoise
############################################

rule nquire_denoise:
    input:
        "results/{ind}.base.bin"
    output:
        "results/{ind}.base_denoised.bin"
    log:
        "results/logs/{ind}_denoise.log"
    resources:
        mem_mb=32000,
        slurm_partition="workq",
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/nQuire/a990a88

        nQuire denoise \
            {input} \
            -o results/{wildcards.ind}.base_denoised \
            &> {log}
        """

############################################
# lrdmodel non denoised (global)
############################################

rule lrdmodel_not_denoised:
    input:
        expand("results/{ind}.base.bin", ind=INDIVIDUALS)
    output:
        "results/all_lrd_not_denoised.txt"
    threads: config["threads_lrdmodel"]
    resources:
        mem_mb=32000,
        slurm_partition="workq",
        time="1-00:00:00"
    log:
        "results/logs/lrd_not_denoised.log"
    shell:
        """
        module load bioinfo/nQuire/a990a88


        nQuire lrdmodel \
            -t {threads} \
            {input} \
            > {output} 2> {log}
        """

############################################
# lrdmodel denoised (global)
############################################

rule lrdmodel_denoised:
    input:
        expand("results/{ind}.base_denoised.bin", ind=INDIVIDUALS)
    output:
        "results/all_lrd_denoised.txt"
    threads: config["threads_lrdmodel"]
    resources:
        mem_mb=32000,
        slurm_partition="workq",
        time="1-00:00:00"
    log:
        "results/logs/lrd_denoised.log"
    shell:
        """
        module load bioinfo/nQuire/a990a88

        nQuire lrdmodel \
            -t {threads} \
            {input} \
            > {output} 2> {log}
        """


############################################
# histogram allele frequency
############################################

rule nquire_histo:
    input:
        "results/{ind}.base_denoised.bin"
    output:
        "results/histo/{ind}.histo.txt"
    log:
        "results/logs/{ind}_histo.log"
    resources:
        mem_mb=8000,
        slurm_partition="workq",
        time="02:00:00"
    shell:
        """
        module load bioinfo/nQuire/a990a88

        mkdir -p results/histo

        nQuire histo \
            {input} \
            > {output} 2> {log}
        """

############################################
# histogram test
############################################

rule nquire_histotest:
    input:
        "results/{ind}.base_denoised.bin"
    output:
        "results/histotest/{ind}.histotest.txt"
    log:
        "results/logs/{ind}_histotest.log"
    resources:
        mem_mb=8000,
        slurm_partition="workq",
        time="02:00:00"
    shell:
        """
        module load bioinfo/nQuire/a990a88

        mkdir -p results/histotest

        nQuire histotest \
            {input} \
            > {output} 2> {log}
        """

In [ ]:
nQuire histo base_denoised.bin
nQuire histotest base_denoised.bin

#### config.yaml L_nobilis_L_azorica

In [ ]:
bam_table: "/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/nQuire/BAM_list.txt"

nquire_module: "bioinfo/nQuire/a990a88"

threads_lrdmodel: 8

#### Run_snakefile_nQuire

In [ ]:
#!/bin/bash
#SBATCH --job-name=nQuire_pipeline_dryrun
#SBATCH -p workq
#SBATCH --time=10:00:00
#SBATCH --mem=2G
#SBATCH --cpus-per-task=1
#SBATCH --mail-type=END,FAIL

module purge
module load bioinfo/Snakemake/8.20.3

mkdir -p results/logs

snakemake \
    --snakefile snakefile_nQuire \
    --rerun-incomplete \
    --keep-going \
    --latency-wait 60 \
    --executor slurm \
    --jobs 12 \
    --cores 12 \
    -n -p


In [ ]:
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-F01Cb_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-F02Cb_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-F03Cb_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-F04Cbf_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-F05Cbf_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-F06Cbf_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-F07b_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-F08Cbf_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-F09Cf_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-F10Cb_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-F11b_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-F12Cf_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-F13Cbf_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-F14Cbf_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-F15Cbf_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-F16b_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-M01Cf_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-M04b_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-M05b_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-M06Cb_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-M07Cf_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-M08Cb_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-M09b_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-M10b_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-M11Cb_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-M12Cb_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-M13Cf_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-M14Cb_TranscriptomeGenomePos.sorted.rg.bam
/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/MAPPING/Lm-M15Cb_TranscriptomeGenomePos.sorted.rg.bam

In [ ]:
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-F01Cb.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-F02Cb.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-F03Cb.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-F04Cbf.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-F05Cbf.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-F06Cbf.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-F07b.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-F08Cbf.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-F09Cf.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-F10Cb.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-F11b.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-F12Cf.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-F13Cbf.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-F14Cbf.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-F15Cbf.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-F16b.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-M01Cf.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-M02Cb.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-M03Cb.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-M04b.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-M05b.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-M06Cb.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-M07Cf.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-M08Cb.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-M09b.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-M10b.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-M11Cb.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-M12Cb.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-M13Cf.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-M14Cb.rg.bam
/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/Lm-M15Cb.rg.bam

# Test REF NOBILIS

scp -r tristan@162.38.181.237:/data2/fabien/L_nobilis/L_nobilis_bam /home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis

#### config.yaml L_nobilis_L_azorica

In [ ]:
bam_table: "/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/L_nobilis_bam/BAM_LIST.txt"
nquire_module: "bioinfo/nQuire/a990a88"
threads_lrdmodel: 8

#### PCA

In [ ]:

# 2) Conversion VCF → PGEN (format natif plink2)



plink2 \
  --vcf /Users/cibio/Desktop/Laurus/Pop_gen/VCF/L_nobilis_L_azorica/L_nobilis_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.vcf.gz \
  --double-id \
  --set-missing-var-ids @:# \
  --allow-extra-chr \
  --snps-only just-acgt \
  --max-alleles 2 \
  --maf 0.000001 \
  --make-pgen \
  --out base_SNP_only

# 3) PCA approx (sans pruning LD/filtrage — juste pour un premier regard)

plink2 \
  --pfile base_SNP_only \
  --read-freq freqs.afreq \
  --pca approx 10 \
  --out pca_quick


plink2 \
  --pfile base_SNP_only \
  --read-freq freqs.afreq \
  --pca approx 10 \
  --out pca_quick


In [ ]:
scp  -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/nQuire/results/all_lrd_denoised.txt /Users/cibio/Desktop/Laurus/nQuire/L_nobilis_L_nobilis
scp  -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis/BAM_L_nobililis_L_nobilis/nQuire/results/all_lrd_not_denoised.txt /Users/cibio/Desktop/Laurus/nQuire/L_nobilis_L_nobilis

scp  -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_azorica/L_azorica_L_azorica/nQuire_TEST/all_lrd_denoised.txt /Users/cibio/Desktop/Laurus/nQuire/L_azorica_L_azorica
scp  -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_azorica/L_azorica_L_azorica/nQuire_TEST/all_lrd_not_denoised.txt /Users/cibio/Desktop/Laurus/nQuire/L_azorica_L_azorica

scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_azorica/L_azorica_L_azorica/GENOTYPING/TEST_FILTERS/STATS/L_azorica_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.het.het /Users/cibio/Desktop/Laurus/nQuire
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/GENOTYPING/STATS/L_nobilis_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.recode.het.het /Users/cibio/Desktop/Laurus/nQuire

scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis/L_nobilis_L_azorica/GENOTYPING/L_nobilis_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.vcf.gz /Users/cibio/Desktop/Laurus/nQuire

scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_azorica/L_azorica_L_azorica/GENOTYPING/L_azorica_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.vcf.gz /Users/cibio/Desktop/Laurus/nQuire